# v4 residual diagnostics

v4 reaches MAE ≈ 0.65 with Optuna-tuned XGBoost on ECFP4 fingerprints. Linus (the dataset author) noted that Krüger et al. (2025) used a graph convolutional NN on this same dataset and only reached **~0.66** — and that the residual ~0.3 log unit gap to e.g. the Wang dataset is likely **dataset noise** from conformational averaging in the COSMOtherm pSat calculations.

Before pouring effort into v5/v6/v7, we want to know: is our remaining 0.65 *model error* (addressable by better features) or *data noise* (irreducible)?

Approach:

1. Train one XGB with v4-style hyperparameters. We don't need the full Optuna sweep — we want the *structure* of the residuals, not the last 0.01 of MAE.
2. Inspect: distribution shape, calibration, dependence on pSat magnitude, dependence on RDKit-computed molecular descriptors.
3. Read what the patterns say:
   - Errors uniformly spread across all axes → likely noise floor. v5 is still worth running (it's a clean test of Linus's specific claim) but expect modest gains.
   - Errors cluster on a property the fingerprint doesn't encode well (size, flexibility, H-bond donor count, …) → real featurisation headroom — exactly what TopFP / ATMOMACCS would target.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

from xgboost import XGBRegressor

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, Lipinski

In [ ]:
data = pd.read_csv(r"C:\Users\ykila\Desktop\iCloud\iCloudDrive\Projects\Active\Code\Python\LLM\Aalto\Dataframe.csv")
data["log_pSat"] = np.log10(data["pSat_Pa"])
print(f"loaded {len(data)} molecules")

In [ ]:
# ECFP4 fingerprints — same featurisation as v2/v3/v4 so this diagnostic
# inherits exactly the representational limitations we're investigating.
def smiles_to_ecfp(smi, n_bits=2048, radius=2):
    arr = np.zeros(n_bits, dtype=np.uint8)
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

X = np.stack([smiles_to_ecfp(s) for s in data["SMILES"]])
y = data["log_pSat"].values
print(f"X shape: {X.shape}")

In [ ]:
# Split tracking indices so we can recover the SMILES for each test row later.
# Same random_state as v4 → X_test here corresponds to v4's X_test exactly.
idx = np.arange(len(X))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42)
idx_tr, idx_val   = train_test_split(idx_train, test_size=0.1, random_state=42)

X_tr,  y_tr   = X[idx_tr],   y[idx_tr]
X_val, y_val  = X[idx_val],  y[idx_val]
X_test, y_test = X[idx_test], y[idx_test]
test_smiles = data.iloc[idx_test]["SMILES"].values

print(f"train {len(idx_tr)} | val {len(idx_val)} | test {len(idx_test)}")

In [ ]:
# Single XGB with v4-style hyperparameters. We're not after v4's best MAE —
# the residual *structure* is what we want, and that's stable across the
# Optuna-search neighbourhood.
xgb = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.6,
    reg_alpha=1.0,
    reg_lambda=1.0,
    gamma=0.1,
    objective="reg:absoluteerror",
    tree_method="hist",
    early_stopping_rounds=50,
    random_state=42,
)
xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

preds = xgb.predict(X_test)
residuals = y_test - preds
abs_residuals = np.abs(residuals)

mae = mean_absolute_error(y_test, preds)
print(f"diagnostic-XGB test MAE = {mae:.4f}   (v4 5-seed bag baseline = 0.6490)")
print(f"best_iteration = {xgb.best_iteration}")

## 1. Residual distribution

Are residuals roughly symmetric and bell-shaped, or are there heavy tails / asymmetry?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(residuals, bins=60, edgecolor="white", linewidth=0.3)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1)
axes[0].set_xlabel("residual = y_true - y_pred (log10 pSat)")
axes[0].set_ylabel("count")
axes[0].set_title("Signed residual distribution")

axes[1].hist(abs_residuals, bins=60, edgecolor="white", linewidth=0.3)
median_ar = np.median(abs_residuals)
axes[1].axvline(median_ar, color="red", linestyle="--", linewidth=1,
                label=f"median = {median_ar:.3f}")
axes[1].set_xlabel("|residual|")
axes[1].set_ylabel("count")
axes[1].set_title("|Residual| distribution")
axes[1].legend()
plt.tight_layout(); plt.show()

## 2. Calibration: predicted vs true

A perfect model lies on `y = x`. Some compression at the extremes (slope < 1) is normal regularisation behaviour for any L1/L2-regularised regressor, **not** model failure. What *would* be failure: clear curvature, separate streaks, or off-diagonal clumps.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, preds, s=4, alpha=0.25)
lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
plt.plot(lims, lims, "r--", linewidth=1, label="y = x")
plt.xlabel("true log10 pSat")
plt.ylabel("predicted log10 pSat")
plt.title("Calibration")
plt.legend()
plt.gca().set_aspect("equal")
plt.tight_layout(); plt.show()

## 3. Errors across the pSat range

Is the |residual| roughly constant in magnitude across the pSat range, or do errors balloon at the extremes? Ballooning at the tails would suggest the model is undertrained on rare extreme values — fixable by reweighting / stratified sampling, independent of featurisation.

In [ ]:
bins = np.linspace(y_test.min(), y_test.max() + 1e-9, 25)
centers = 0.5 * (bins[:-1] + bins[1:])
binned_mae = np.array([
    abs_residuals[(y_test >= lo) & (y_test < hi)].mean()
        if ((y_test >= lo) & (y_test < hi)).any() else np.nan
    for lo, hi in zip(bins[:-1], bins[1:])
])

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(y_test, abs_residuals, s=3, alpha=0.2)
ax.plot(centers, binned_mae, color="red", linewidth=2, label="binned mean |residual|")
ax.set_xlabel("true log10 pSat")
ax.set_ylabel("|residual|")
ax.set_title("|Residual| across the pSat range")
ax.legend()
plt.tight_layout(); plt.show()

## 4. Do errors correlate with molecular complexity?

**This is the diagnostic-deciding plot for v5/v6 expectations.** For each test molecule we compute a handful of RDKit descriptors and ask: does |residual| correlate with any of them? A non-trivial Spearman ρ flags a property that ECFP4 doesn't encode well — i.e. real featurisation headroom that swapping or adding fingerprints could unlock.

In [ ]:
def compute_descriptors(smiles_list):
    rows = []
    for s in smiles_list:
        m = Chem.MolFromSmiles(s)
        if m is None:
            rows.append([np.nan] * 7); continue
        rows.append([
            Descriptors.HeavyAtomCount(m),
            Descriptors.MolWt(m),
            Descriptors.NumRotatableBonds(m),
            Lipinski.NumHDonors(m),
            Lipinski.NumHAcceptors(m),
            Descriptors.RingCount(m),
            Descriptors.TPSA(m),
        ])
    return pd.DataFrame(rows, columns=[
        "heavy_atoms", "MW", "rotatable_bonds", "HBD", "HBA", "rings", "TPSA"
    ])

desc_df = compute_descriptors(test_smiles)
desc_df.head()

In [ ]:
cols = ["heavy_atoms", "MW", "rotatable_bonds", "HBD", "HBA", "rings", "TPSA"]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
spearman = []
for ax, col in zip(axes.flat, cols):
    vals = desc_df[col].values
    rho, p = spearmanr(vals, abs_residuals)
    spearman.append((col, rho, p))
    ax.scatter(vals, abs_residuals, s=3, alpha=0.2)
    ax.set_xlabel(col)
    ax.set_ylabel("|residual|")
    ax.set_title(f"{col}   (ρ = {rho:+.3f})")
axes.flat[7].axis("off")
plt.tight_layout(); plt.show()

print("\nSpearman ρ between |residual| and descriptor:\n")
for col, rho, p in spearman:
    flag = "  <-- nontrivial" if abs(rho) > 0.15 else ""
    print(f"  {col:18s}  ρ = {rho:+.3f}   p = {p:.2e}{flag}")

## 5. The worst predictions

The 20 test molecules with the largest |residual|. Eyeball the SMILES: do they share an obvious chemotype, lots of flexible chains, unusual heteroatom patterns? Patterns here are the most concrete clue for what featurisation should target.

In [ ]:
worst = pd.DataFrame({
    "SMILES": test_smiles,
    "y_true": y_test,
    "y_pred": preds,
    "abs_error": abs_residuals,
    "heavy_atoms": desc_df["heavy_atoms"].values,
    "rotatable_bonds": desc_df["rotatable_bonds"].values,
    "HBD": desc_df["HBD"].values,
    "HBA": desc_df["HBA"].values,
}).sort_values("abs_error", ascending=False).head(20).reset_index(drop=True)

print(worst.to_string(index=False))

## Reading the diagnostics

**Residual histogram (§1).** A symmetric, roughly bell-shaped distribution with thin tails is consistent with many small uncorrelated error sources — i.e. noise. Heavy or asymmetric tails suggest systematic failure modes worth investigating.

**Calibration (§2).** Slight slope < 1 at the extremes is normal regularisation, not failure. Real failure looks like clear curvature, separate streaks, or off-diagonal clumps.

**|Residual| vs pSat range (§3).** Roughly flat → errors don't depend on pSat magnitude (noise-floor pattern). Ballooning at the tails → undertrained extremes (fixable by reweighting / target transform / stratified sampling, independent of featurisation).

**Spearman ρ vs descriptors (§4)** — *the most diagnostic plot for v5/v6 expectations:*

- All |ρ| ≲ 0.05 → ECFP4 already captures the signal these descriptors carry. Featurisation changes will likely give modest gains; the remaining gap is mostly data noise.
- |ρ| > 0.15 on `rotatable_bonds` or `MW` → ECFP4 is missing chain-length / size signal. **TopFP with high `maxPath`** (Linus's specific recommendation) targets exactly this — linear paths up to length 7 encode topology that radius-2 circular fragments can't.
- |ρ| > 0.15 on `HBD` / `HBA` / `TPSA` → fingerprints alone miss H-bonding character. Concatenating RDKit descriptors as auxiliary features (Linus's "combine different molecular features" suggestion) is the right intervention.

**Worst predictions (§5).** Common chemotype across the worst 20 → featurisation gap. Random-looking → noise.

Once you've inspected these, you'll know whether v5 (TopFP swap) is high-headroom or low-headroom-but-still-worth-running confirmation of Linus's claim.